## Torso Parallel mechanism

## Inverse Kinematics

In [ ]:
import sympy as sp
import numpy as np

ls, lc, lr, lo, ld = sp.symbols('ls lc lr lo ld', positive=True, real=True)
alpha, beta = sp.symbols('alpha beta', real=True)
theta1, theta2 = sp.symbols('theta1 theta2', real=True)

# Rotation matrices
def rotX(theta):
    return sp.Matrix([
        [1, 0, 0],
        [0, sp.cos(theta), -sp.sin(theta)],
        [0, sp.sin(theta),  sp.cos(theta)]
    ])
def rotY(theta):
    return sp.Matrix([
        [sp.cos(theta), 0, sp.sin(theta)],
        [0, 1, 0],
        [-sp.sin(theta), 0, sp.cos(theta)]
    ])
def rot_axis(axis, angle):
    axis = sp.Matrix(axis)
    axis = axis / sp.sqrt(axis.dot(axis))
    x, y, z = axis
    c = sp.cos(angle)
    s = sp.sin(angle)
    return sp.Matrix([
        [c + x*x*(1 - c),     x*y*(1 - c) - z*s, x*z*(1 - c) + y*s],
        [y*x*(1 - c) + z*s,   c + y*y*(1 - c),   y*z*(1 - c) - x*s],
        [z*x*(1 - c) - y*s,   z*y*(1 - c) + x*s, c + z*z*(1 - c)]
    ])
def clean(expr):
    # expr = sp.expand_trig(expr)
    # expr = sp.together(expr)
    expr = sp.cancel(expr)
    expr = sp.trigsimp(expr)
    expr = sp.simplify(expr)
    return expr

# Home positions
r_o_a2_home = sp.Matrix([ 0,    ls/2,   ld])
r_o_b2_home = sp.Matrix([ lc,   ls/2,   ld])
r_o_c2_home = sp.Matrix([ lc,   ls/2,  -lo])

r_o_a3_home = sp.Matrix([ 0,    -ls/2,   ld])
r_o_b3_home = sp.Matrix([ lc,   -ls/2,   ld])
r_o_c3_home = sp.Matrix([ lc,   -ls/2,  -lo])

### Base → Torso rotation

In [164]:
rot_base_torso = rotY(beta) * rotX(alpha)

# Intermediate position
r_o_a2_int = rot_base_torso * r_o_a2_home 
r_o_b2_int = rot_base_torso * r_o_b2_home
r_o_a3_int = rot_base_torso * r_o_a3_home 
r_o_b3_int = rot_base_torso * r_o_b3_home

### Loop closure equation

In [165]:
motor_axis = r_o_a2_int - r_o_a3_int
u = motor_axis / motor_axis.norm()

# Vectors R1, R2
R1 = r_o_c2_home - r_o_a2_int
R2 = r_o_b2_int - r_o_a2_int

R2_prll = u * (u.dot(R2))
R2_perp = R2 - R2_prll

### Analytical solution 

$$
A\cos\theta + B\sin\theta = C
$$

In [166]:
a = R1.dot(R2_perp)
b = R1.dot(u.cross(R2))
c = ((R1.dot(R1)) + (R2.dot(R2)) - lr**2)/2.0 - R1.dot(R2_prll)

In [167]:
clean(a)


lc*(lc*cos(beta) + lo*sin(beta))

In [168]:
clean(b)


lc*(-2*lc*sin(beta)*cos(alpha) + 2*ld + 2*lo*cos(alpha)*cos(beta) + ls*sin(alpha))/2

In [169]:
clean(c)

1.0*lc**2 - 1.0*lc*ld*sin(beta)*cos(alpha) - 0.5*lc*ls*sin(alpha)*sin(beta) + 0.5*ld**2 + 1.0*ld*lo*cos(alpha)*cos(beta) + 0.5*ld*ls*sin(alpha) + 0.5*lo**2 + 0.5*lo*ls*sin(alpha)*cos(beta) - 0.5*lr**2 - 0.25*ls**2*cos(alpha) + 0.25*ls**2

## Analytical solution

In [170]:
a = lc*(lc*sp.cos(beta) + lo*sp.sin(beta))
b = lc*(-2*lc*sp.sin(beta)*sp.cos(alpha) + 2*ld + 2*lo*sp.cos(alpha)*sp.cos(beta) + ls*sp.sin(alpha))/2
c = 1.0*lc**2 - 1.0*lc*ld*sp.sin(beta)*sp.cos(alpha) - 0.5*lc*ls*sp.sin(alpha)*sp.sin(beta) + 0.5*ld**2 + 1.0*ld*lo*sp.cos(alpha)*sp.cos(beta) + 0.5*ld*ls*sp.sin(alpha) + 0.5*lo**2 + 0.5*lo*ls*sp.sin(alpha)*sp.cos(beta) - 0.5*lr**2 - 0.25*ls**2*sp.cos(alpha) + 0.25*ls**2

def theta_values(alpha_deg, beta_deg):
    base = {
        lc: 0.075,
        lr: 0.115,
        lo: 0.025,
        ld: 0.09,
        alpha: alpha_deg * sp.pi/180,
        beta:  beta_deg  * sp.pi/180
    }

    def solve(ls_val):
        vals = base | {ls: ls_val}
        a_v = float(a.subs(vals))
        b_v = float(b.subs(vals))
        c_v = float(c.subs(vals))

        D = a_v*a_v + b_v*b_v
        disc = b_v*b_v*c_v*c_v - D*(c_v*c_v - a_v*a_v)
        r = np.sqrt(disc)

        s = (b_v*c_v - r) / D
        c_ = (c_v - b_v*s) / a_v
        return np.arctan2(s, c_)

    return solve(0.08), solve(-0.08)

theta1, theta2 = theta_values(5.0, 10.0)    # roll,pitch

print(f"theta1: {np.degrees(theta1):.6f}°")
print(f"theta2: {np.degrees(theta2):.6f}°")


theta1: -7.578045°
theta2: -12.957342°


## Loop Closure Equation
### Constraint equation

$$
\|\mathbf{R}_1 - \mathrm{Rot}(\theta)\,\mathbf{R}_2\|^2 = l_r^2
\qquad (1)
$$

where

$$
\mathbf{R}_1 = \mathbf{r}_C - \mathbf{r}_A
$$

$$
\mathbf{R}_2 = \mathbf{r}_B - \mathbf{r}_A
$$

---

### Rodrigues rotation formula

By Rodrigues’ rotation formula,

$$
\mathrm{Rot}(\theta)\,\mathbf{R}_2
=
\mathbf{R}_{2\parallel}
+
\mathbf{R}_{2\perp}\cos\theta
+
(\mathbf{u}\times\mathbf{R}_2)\sin\theta
$$

---

### Substitution into constraint equation

Substituting into (1),

$$
\|\mathbf{R}_1 - \mathrm{Rot}(\theta)\,\mathbf{R}_2\|^2
=
\mathbf{R}_1\cdot\mathbf{R}_1
+
\mathbf{R}_2\cdot\mathbf{R}_2
-
2\,\mathbf{R}_1\cdot\mathrm{Rot}(\theta)\mathbf{R}_2
$$

Thus,

$$
\mathbf{R}_1\cdot\mathrm{Rot}(\theta)\mathbf{R}_2
=
\frac{\mathbf{R}_1\cdot\mathbf{R}_1
+
\mathbf{R}_2\cdot\mathbf{R}_2
-
l_r^2}{2}
$$

---

### Expanding the dot product

$$
\mathbf{R}_1\cdot\mathbf{R}_{2\parallel}
+
\mathbf{R}_1\cdot\mathbf{R}_{2\perp}\cos\theta
+
\mathbf{R}_1\cdot(\mathbf{u}\times\mathbf{R}_2)\sin\theta
=
\frac{\mathbf{R}_1\cdot\mathbf{R}_1
+
\mathbf{R}_2\cdot\mathbf{R}_2
-
l_r^2}{2}
$$

Rearranging,

$$
\mathbf{R}_1\cdot\mathbf{R}_{2\perp}\cos\theta
+
\mathbf{R}_1\cdot(\mathbf{u}\times\mathbf{R}_2)\sin\theta
=
\frac{\mathbf{R}_1\cdot\mathbf{R}_1
+
\mathbf{R}_2\cdot\mathbf{R}_2
-
l_r^2}{2}
-
\mathbf{R}_1\cdot\mathbf{R}_{2\parallel}
\qquad (2)
$$

---

### Standard trigonometric form

Equation (2) is of the form

$$
A\cos\theta + B\sin\theta = C
$$

where

$$
A = \mathbf{R}_1\cdot\mathbf{R}_{2\perp}
$$

$$
B = \mathbf{R}_1\cdot(\mathbf{u}\times\mathbf{R}_2)
$$

$$
C =
\frac{\mathbf{R}_1\cdot\mathbf{R}_1
+
\mathbf{R}_2\cdot\mathbf{R}_2
-
l_r^2}{2}
-
\mathbf{R}_1\cdot\mathbf{R}_{2\parallel}
$$


## Torque Requirement

In [171]:
m_torso = 30
m_arm = 14
payload = 7.5
l_arm = 0.65
l_torso = 0.45

pitch_angle = 15

pitch_torque = (
    (m_arm * 2 * ((l_arm/2) + (l_torso * np.sin(np.deg2rad(pitch_angle))))) + 
    (m_torso * np.sin(np.deg2rad(pitch_angle)) *(l_torso/2)) + 
    (payload * 2 * ((l_arm + 0.1) + (l_torso * np.sin(np.deg2rad(pitch_angle)))))
    ) * 9.81
pitch_dynamic = pitch_torque * 1.2

print(f"Torque for each actuator in the parallel mechanism: {pitch_dynamic/2}")
print(f"Torque for pitching: {pitch_dynamic}")

Torque for each actuator in the parallel mechanism: 159.5410722762567
Torque for pitching: 319.0821445525134
